# SC-SSTW RC1 matched-triplet execution package

Preparation only: this notebook does not embed authorization, data, or credentials and does not authorize any later stage. A qualifying frozen L1-v2 selected-candidate package is checked before generation. The formal topology always uses --generate so production evidence is observed and validated in the same runner process; an external execution package is never accepted here. All failures produce a local audit/log archive; only a re-read and verified Drive copy is reported as packaged.

In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, sys, traceback, zipfile
LOCAL_ROOT = pathlib.Path('/content/sc-sstw-rc1')
LOG_ROOT = pathlib.Path('/content/sc-sstw-rc1-logs')
LOG_ROOT.mkdir(parents=True, exist_ok=True)
EVENTS = LOG_ROOT / 'notebook_events.jsonl'
def event(stage, **fields):
    with EVENTS.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps({'stage': stage, **fields}, sort_keys=True) + '\n')
def run(command, cwd=None, allowed_returncodes=(0,)):
    event('command_start', command=command, cwd=str(cwd) if cwd else None)
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    with (LOG_ROOT / 'stdout.log').open('a', encoding='utf-8') as handle: handle.write(result.stdout)
    with (LOG_ROOT / 'stderr.log').open('a', encoding='utf-8') as handle: handle.write(result.stderr)
    if result.returncode not in allowed_returncodes: raise RuntimeError(f'command failed ({result.returncode}): {command}')
    return result
def load_runner_audit(result, audit_path):
    if not audit_path.is_file(): raise RuntimeError('runner audit is missing')
    try: audit = json.loads(audit_path.read_text(encoding='utf-8'))
    except Exception as exc: raise RuntimeError('runner audit is malformed') from exc
    required = {'schema_version', 'output_schema', 'protocol_id', 'status', 'valid_experiment', 'reason_code', 'formal_result', 'stage_progression_allowed'}
    if not required.issubset(audit): raise RuntimeError('runner audit schema is incomplete')
    if audit['schema_version'] != 1 or audit['output_schema'] != 'sc_sstw_rc1_method_validation_audit_v1' or audit['protocol_id'] != 'sc_sstw_rc1_saved_mp4_relation_validation': raise RuntimeError('runner audit identity changed')
    if audit['formal_result'] is not False or audit['stage_progression_allowed'] is not False: raise RuntimeError('runner audit evidence boundary changed')
    expected = {'RC1_VALID_PASS': (0, True), 'RC1_VALID_FAIL': (3, True), 'PREREQUISITE_NOT_MET': (2, False), 'INVALID_EXPERIMENT': (2, False)}
    if audit['status'] not in expected or (result.returncode, audit['valid_experiment']) != expected[audit['status']]: raise RuntimeError('runner exit/status semantics are inconsistent')
    return audit
def sha256(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): digest.update(chunk)
    return digest.hexdigest()
event('local_logging_ready')

In [ ]:
def checkout_exact_source():
    repository_url = os.environ.get('SC_SSTW_RC1_REPOSITORY_URL', '').strip()
    exact_ref = os.environ.get('SC_SSTW_RC1_EXACT_REF', '').strip()
    if not repository_url or len(exact_ref) != 40 or any(c not in '0123456789abcdef' for c in exact_ref):
        raise RuntimeError('explicit repository URL and exact 40-hex SC_SSTW_RC1_EXACT_REF are required; main is never implied')
    run(['git', 'clone', '--no-checkout', repository_url, str(LOCAL_ROOT)])
    run(['git', 'fetch', 'origin', exact_ref], cwd=LOCAL_ROOT)
    run(['git', 'checkout', '--detach', exact_ref], cwd=LOCAL_ROOT)
    actual_head = run(['git', 'rev-parse', 'HEAD'], cwd=LOCAL_ROOT).stdout.strip()
    actual_tree = run(['git', 'rev-parse', 'HEAD^{tree}'], cwd=LOCAL_ROOT).stdout.strip()
    dirty = run(['git', 'status', '--porcelain=v1', '--untracked-files=all'], cwd=LOCAL_ROOT).stdout.strip()
    if actual_head != exact_ref or dirty: raise RuntimeError('checkout identity or clean-state check failed')
    event('checkout_verified', head=actual_head, tree=actual_tree, dirty=False)
    return exact_ref

In [ ]:
def mount_and_identify_inputs():
    from google.colab import drive
    drive.mount('/content/drive')
    manifest_raw = os.environ.get('SC_SSTW_RC1_AUTHORIZATION_MANIFEST', '').strip()
    prerequisite_raw = os.environ.get('SC_SSTW_RC1_PREREQUISITE_PACKAGE', '').strip()
    drive_output_raw = os.environ.get('SC_SSTW_RC1_DRIVE_OUTPUT', '').strip()
    if not manifest_raw or not prerequisite_raw or not drive_output_raw: raise RuntimeError('explicit manifest, prerequisite, and Drive output paths are required')
    manifest_path = pathlib.Path(manifest_raw)
    prerequisite = pathlib.Path(prerequisite_raw)
    drive_output = pathlib.Path(drive_output_raw)
    for name, path in [('manifest', manifest_path), ('prerequisite', prerequisite), ('Drive output', drive_output)]:
        if any(str(i) in str(path) for i in range(41001, 41009)): raise RuntimeError(f'explicit non-formal {name} path is required')
    if not manifest_path.is_file() or not prerequisite.is_dir(): raise RuntimeError('manifest or prerequisite package is unavailable')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest.get('evidence_policy') != {'default_mode': 'production_saved_mp4', 'synthetic_fixture_permitted': False}: raise RuntimeError('formal notebook requires production-only manifest evidence policy')
    declared_package = pathlib.Path(manifest['prerequisite']['package_path'])
    resolved_package = declared_package.resolve() if declared_package.is_absolute() else (manifest_path.parent / declared_package).resolve()
    if resolved_package != prerequisite.resolve(): raise RuntimeError('manifest does not bind the explicitly supplied prerequisite path')
    event('external_inputs_identified', manifest_sha256=sha256(manifest_path), prerequisite_checksums_sha256=sha256(prerequisite / 'checksums.sha256'))
    return manifest_path, prerequisite, drive_output

In [ ]:
def prepare_runtime():
    locked = ['accelerate==1.4.0', 'diffusers==0.35.2', 'ftfy==6.3.1', 'imageio==2.37.0', 'huggingface_hub==0.35.3', 'imageio_ffmpeg==0.6.0', 'numpy==1.26.4', 'safetensors==0.5.3', 'transformers==4.49.0']
    run([sys.executable, '-m', 'pip', 'install', *locked], cwd=LOCAL_ROOT)
    import torch
    if not torch.cuda.is_available(): raise RuntimeError('authorized execution requires a CUDA runtime')
    runtime = {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': torch.version.cuda}
    (LOG_ROOT / 'runtime.json').write_text(json.dumps(runtime, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    event('runtime_verified', **runtime)

In [ ]:
RESULT_ROOT = pathlib.Path('/content/sc-sstw-rc1-result')
DRIVE_OUTPUT = None
packaged = False
try:
    EXACT_REF = checkout_exact_source()
    MANIFEST, PREREQUISITE, DRIVE_OUTPUT = mount_and_identify_inputs()
    prepare_runtime()
    command = [sys.executable, 'experiments/run_rc1_method_validation.py', '--manifest', str(MANIFEST), '--output', str(RESULT_ROOT), '--source-commit', EXACT_REF, '--generate']
    result = run(command, cwd=LOCAL_ROOT, allowed_returncodes=(0, 2, 3))
    audit = load_runner_audit(result, RESULT_ROOT / 'audit.json')
    event('runner_finished', status=audit['status'], reason_code=audit['reason_code'], returncode=result.returncode, nonzero_preserved=result.returncode != 0)
except Exception as exc:
    event('runner_exception', exception_type=type(exc).__name__, detail=str(exc), traceback=traceback.format_exc())
    raise
finally:
    archive_base = pathlib.Path('/content/sc-sstw-rc1-package')
    staging = pathlib.Path('/content/sc-sstw-rc1-package-staging')
    staging.mkdir(parents=True, exist_ok=True)
    if RESULT_ROOT.exists(): shutil.copytree(RESULT_ROOT, staging / 'result', dirs_exist_ok=True)
    shutil.copytree(LOG_ROOT, staging / 'notebook_logs', dirs_exist_ok=True)
    archive = pathlib.Path(shutil.make_archive(str(archive_base), 'zip', staging))
    local_size, local_sha = archive.stat().st_size, sha256(archive)
    if DRIVE_OUTPUT is not None:
        DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
        drive_copy = DRIVE_OUTPUT / archive.name
        shutil.copy2(archive, drive_copy)
        drive_size, drive_sha = drive_copy.stat().st_size, sha256(drive_copy)
        with zipfile.ZipFile(drive_copy, 'r') as handle: bad_member = handle.testzip()
        if (drive_size, drive_sha, bad_member) != (local_size, local_sha, None): raise RuntimeError('Drive-local ZIP readback verification failed')
        verification = {'packaged': True, 'path': str(drive_copy), 'size': drive_size, 'sha256': drive_sha, 'zip_readable': True}
        verification_path = DRIVE_OUTPUT / (archive.name + '.verification.json')
        verification_path.write_text(json.dumps(verification, indent=2, sort_keys=True) + '\n', encoding='utf-8')
        if json.loads(verification_path.read_text(encoding='utf-8')) != verification: raise RuntimeError('Drive verification sidecar readback failed')
        packaged = True
        event('drive_copy_verified', **verification)
print(json.dumps({'packaged': packaged, 'local_archive': str(archive), 'local_sha256': local_sha, 'local_size': local_size}, sort_keys=True))

The notebook stops after verified packaging. It does not authorize or start a later phase.